# tpcve — оценка биомассы пшеницы (Kaggle)

Запуск трёх методов (`voxel`, `chm`, `alpha`) по dataset-спискам, вместе и раздельно по стадиям роста `--stage Z31` (дата 0828) и `--stage Z65` (дата 1002).

Каждый запуск: batch (признак по облакам) → автоматический analyze (регрессия `biomass ~ признак`, train+test).

**Требования:** в настройках ноутбука включить **Internet ON** (нужно для `git clone`) и подключить датасет `kobanur/wheat-biomass-and-lidar-dataset-yanco-2019`.

Результаты пишутся в `/kaggle/working/tpcve/results/` (volume CSV, regression CSV, графики).

## 1. Клонирование кода

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Konabur/tpcve"
REPO_DIR = "/kaggle/working/tpcve"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("repo already cloned")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 2. Установка зависимостей

`pip install -e .` ставит пакет editable и тянет зависимости из `pyproject.toml` (open3d, alphashape, scikit-learn, python-dotenv и т.д.). Импорты `import tpcve` / `import tools` работают из любого cwd.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", "."], check=True)
print("deps installed")

## 3. Пути к данным + проверка

Списки лежат в `DATA`, пути внутри списков относительны `DATA` → `--base-dir DATA`. Сначала убеждаемся, что первый файл из train-списка реально находится.

In [ ]:
from pathlib import Path
from tpcve.core.io import parse_list_line

DATA = Path("/kaggle/input/datasets/kobanur/wheat-biomass-and-lidar-dataset-yanco-2019/data/Yanco_TC_2019_HI-pcd")
TRAIN_LIST = DATA / ".." / "train_list.txt"
TEST_LIST  = DATA / ".." / "test_list.txt"

for p in (TRAIN_LIST, TEST_LIST):
    assert p.exists(), f"нет файла: {p}"

# sanity-check: первая валидная строка train-списка -> существующий .pcd
with open(TRAIN_LIST, encoding="utf-8") as f:
    for line in f:
        if line.strip() and not line.lstrip().startswith("#"):
            rel, labels = parse_list_line(line)
            full = DATA / rel.lstrip("/\\")
            print("rel  :", rel)
            print("full :", full)
            print("exists:", full.exists(), "| biomass:", labels["biomass"])
            assert full.exists(), "base-dir не совпадает с путями из списка — поправь DATA"
            break

In [ ]:
def run_batch(method, stage=None, *method_flags, **kwargs):
    """Запустить batch.py для одного метода/стадии (train+test, analyze в цепочке)."""
    cmd = [
        sys.executable, "batch.py",
        "--method", method,
        "--list", str(TRAIN_LIST),
        "--list-test", str(TEST_LIST),
        "--base-dir", str(DATA),
        "--workers", "4",
        *method_flags,
    ]
    if stage is not None:
        cmd.extend(["--stage", stage])
    print("\n$ " + " ".join(cmd) + "\n")
    p = subprocess.run(cmd, cwd=REPO_DIR, **kwargs)
    if p.returncode:
        raise SystemExit(f"batch завершился с кодом {p.returncode}")

In [ ]:
STAGES = [None, "Z31", "Z65"]

# STAGES = [None]          # только combined
# STAGES = ["Z31"]         # только Z31
# STAGES = ["Z65"]         # только Z65
# STAGES = ["Z31", "Z65"]  # раздельно две стадии

## 4. Воксельный метод

`s (--voxel-sizes): 10, 15, 20, 25, 30, 35, 40, 45` (мм)

In [ ]:
for s in STAGES:
    run_batch("voxel", s, "--voxel-sizes", "10,15,20,25,30,35,40,45")

## 5. CHM (объём по сетке высот)

`c (--cell-sizes): 10, 15, 20, 25` (мм) · `p (--percentiles): 1, 5, 10, 50, 75, 95, 99`

In [ ]:
for s in STAGES:
    run_batch("chm", s, "--cell-sizes", "10,15,20,25", "--percentiles", "1,5,10,50,75,95,99")

## 6. Послойная альфа-форма

`v (--voxel-sizes): 10, 20, 30, 40, 50` (мм) · `α (--alphas): 20, 30, 40` · `dz (--layer-dz): 10, 20, 30, 40, 50, 60` (мм)

In [ ]:
for s in STAGES:
    run_batch("alpha", s,
              "--voxel-sizes", "10,20,30,40,50",
              "--alphas", "20,30,40",
              "--layer-dz", "10,20,30,40,50,60",
              "--no-inner-progress",)

## 7. Бейзлайны

`count` — число точек растительности → биомасса (простейший baseline). `percentile` — один скаляр (перцентиль высоты) → биомасса.

In [ ]:
for s in STAGES:
    run_batch("count", s)

In [ ]:
for s in STAGES:
    run_batch("percentile", s, "--percentiles", "1,5,10,50,75,95,99")

## 8. Результаты

Все CSV и графики — в `results/`. Ниже список свежих regression-CSV (отсортированы по R²) и упаковка в один архив для скачивания.

In [ ]:
import shutil

results_dir = Path(REPO_DIR) / "results"
print("regression CSV:")
for p in sorted(results_dir.glob("regression_csv/**/*.csv")):
    print("  ", p.relative_to(results_dir))

archive = shutil.make_archive("/kaggle/working/tpcve_results", "zip", results_dir)
print("\narchive:", archive)

## 9. Предсказание биомассы (демо)

Для каждого варианта (`combined` / `Z31` / `Z65`) берёт train-регрессию (CSV без `_test`) соответствующей стадии по всем методам (`voxel`/`alpha`/`chm`/`percentile`/`count`), выбирает медианное по биомассе облако из `TEST_LIST`, прогоняет на нём все методы и печатает таблицу с абс./отн. ошибкой.

Запускай **после** того, как соответствующие batch+analyze отработали (combined и обе стадии).

In [ ]:
reg = results_dir / "regression_csv"

def pick_csv(method, stage):
    cands = [p for p in (reg / method).glob("*.csv")
             if not p.stem.endswith("_test")
             and (f"_{stage}_" in f"_{p.stem}_" if stage
                  else ("_Z31_" not in p.stem and "_Z65_" not in p.stem))]
    assert len(cands) == 1, f"{method}/{stage}: ожидал 1 CSV, нашёл {len(cands)}: {cands}"
    return cands[0]

for stage in STAGES:
    print("\n" + "=" * 70)
    print(f"=== predict: stage={stage or 'combined'} ===")
    print("=" * 70)
    cmd = [
        sys.executable, "scripts/predict_biomass.py",
        "--list", str(TEST_LIST),
        "--base-dir", str(DATA),
        "--voxel-csv",  str(pick_csv("voxel", stage)),
        "--alpha-csv",  str(pick_csv("alpha", stage)),
        "--chm-csv",    str(pick_csv("chm", stage)),
        "--height-csv", str(pick_csv("percentile", stage)),
        "--count-csv",  str(pick_csv("count", stage)),
    ]
    print("\n$ " + " ".join(cmd) + "\n")
    p = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(p.stdout, end="")
    if p.stderr:
        print(p.stderr, end="")
    if p.returncode:
        raise SystemExit(f"predict (stage={stage}) завершился с кодом {p.returncode}")

## 10. Сводные графики

Горизонтальные bar chart: лучший R² + bootstrap CI для каждого метода, раздельно по стадиям.

In [ ]:
from IPython.display import Image, display

summary_dir = results_dir / "figures" / "summary"

# 1. Сгенерировать сводные графики
subprocess.run([
    sys.executable, "scripts/plot_summary.py",
    "--results-dir", str(results_dir),
    "--output-dir", str(summary_dir),
], cwd=REPO_DIR, check=True)

# 2. Показать per-stage bar charts
for s in STAGES:
    png = summary_dir / f"r2_stage_{s}.png"
    if png.exists():
        print(f"\nStage: {s}")
        display(Image(filename=str(png)))

# 3. Показать лучшие регрессионные графики
best_dir = summary_dir / "best_fits"
if best_dir.is_dir():
    for png in sorted(best_dir.glob("*.png")):
        print(f"\n{png.stem}")
        display(Image(filename=str(png)))